This is a p good model for maybe the top end analysis. Seems to be doing pretty well

In [31]:
from transformers import pipeline

# Load the classification pipeline with the specified model
pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis", return_all_scores=False)

# Classify a new sentence
sentence = "I may have been too much, I think I might have stunk myself"
result = pipe(sentence)

# Print the result
print(result)

[{'label': 'Negative', 'score': 0.8532438278198242}]


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\pipelines\text_classification.py:104: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


Interest model: https://huggingface.co/firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3

Defo needs more testing, but interesting. It generalizes the whole entry tho. This will need some way to parse sentences which is weird. Maybe this can be used as an over-the-top model? idk

In [33]:
# Requires: librosa
from transformers import AutoModelForAudioClassification, AutoFeatureExtractor
import librosa
import torch
import numpy as np

model_id = "firdhokk/speech-emotion-recognition-with-openai-whisper-large-v3"
model = AutoModelForAudioClassification.from_pretrained(model_id)

feature_extractor = AutoFeatureExtractor.from_pretrained(model_id, do_normalize=True)
id2label = model.config.id2label

def preprocess_audio(audio_path, feature_extractor, max_duration=30.0):
    audio_array, sampling_rate = librosa.load(audio_path, sr=None)
    
    max_length = int(feature_extractor.sampling_rate * max_duration)
    if len(audio_array) > max_length:
        audio_array = audio_array[:max_length]
    else:
        audio_array = np.pad(audio_array, (0, max_length - len(audio_array)))

    inputs = feature_extractor(
        audio_array,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=max_length,
        truncation=True,
        return_tensors="pt",
    )
    return inputs

def predict_emotion(audio_path, model, feature_extractor, id2label, max_duration=30.0):
    inputs = preprocess_audio(audio_path, feature_extractor, max_duration)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_id = torch.argmax(logits, dim=-1).item()
    predicted_label = id2label[predicted_id]
    
    return predicted_label

audio_path = r"C:\Users\User\projects\MindLine\testing_models\test_audios\Test3.wav"

predicted_emotion = predict_emotion(audio_path, model, feature_extractor, id2label)
print(f"Predicted Emotion: {predicted_emotion}")


Predicted Emotion: angry


Sarcasm detector cuz sarcasm is still huge

This one below is kinda mid

In [8]:
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
import string

def preprocess_data(text: str) -> str:
   return text.lower().translate(str.maketrans("", "", string.punctuation)).strip()

MODEL_PATH = "helinivan/english-sarcasm-detector"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

text = "Yeah, I love the pain"
tokenized_text = tokenizer([preprocess_data(text)], padding=True, truncation=True, max_length=256, return_tensors="pt")
output = model(**tokenized_text)
probs = output.logits.softmax(dim=-1).tolist()[0]
confidence = max(probs)
prediction = probs.index(confidence)
results = {"is_sarcastic": prediction, "confidence": confidence}
print(results)


c:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


{'is_sarcastic': 0, 'confidence': 0.896741509437561}


I think we found the one for emotions gang LESGO

In [43]:
from transformers import pipeline

classifier = pipeline(task="text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None)

sentences = ["Today was a great day, man, like I answered really well in all my classes and even got all my homework done on time."]

model_outputs = classifier(sentences)
model_outputs[0]
# produces a list of dicts for each of the labels


[{'label': 'admiration', 'score': 0.7527955770492554},
 {'label': 'joy', 'score': 0.20656147599220276},
 {'label': 'approval', 'score': 0.190708190202713},
 {'label': 'excitement', 'score': 0.050287846475839615},
 {'label': 'gratitude', 'score': 0.04175783321261406},
 {'label': 'neutral', 'score': 0.03310764953494072},
 {'label': 'pride', 'score': 0.032323207706213},
 {'label': 'relief', 'score': 0.01693177968263626},
 {'label': 'optimism', 'score': 0.013518749736249447},
 {'label': 'realization', 'score': 0.008861825801432133},
 {'label': 'caring', 'score': 0.006696018390357494},
 {'label': 'annoyance', 'score': 0.005733428988605738},
 {'label': 'desire', 'score': 0.0046125552617013454},
 {'label': 'love', 'score': 0.0033356572967022657},
 {'label': 'amusement', 'score': 0.0033222553320229053},
 {'label': 'disapproval', 'score': 0.0032551479525864124},
 {'label': 'disappointment', 'score': 0.0019384550396353006},
 {'label': 'surprise', 'score': 0.001767786219716072},
 {'label': 'anger

Testing out the entity detection options

In [56]:
import spacy

sentences = [
    'Today was a great day, man—like I answered really well in all my classes and even got all my Calculus-II homework done on time.', 
    'On top of that, I hit it off with this guy named Shawn and had a lot of fun.', 
    'I think Shawn is a great dude, and Shawn and I would make amazing friends—like, how can this day get any better?'
]

nlp = spacy.load("en_core_web_sm")

for text in sentences:
    doc = nlp(text)
    
    # Iterate through every individual word/token
    for token in doc:
        if token.pos_ in ["PROPN", "NOUN"]:
            print(f"{token.text: <15} | Tag: {token.pos_}")

Today           | Tag: NOUN
day             | Tag: NOUN
man             | Tag: NOUN
classes         | Tag: NOUN
Calculus        | Tag: PROPN
II              | Tag: PROPN
homework        | Tag: NOUN
time            | Tag: NOUN
top             | Tag: NOUN
guy             | Tag: NOUN
Shawn           | Tag: PROPN
lot             | Tag: NOUN
fun             | Tag: NOUN
Shawn           | Tag: PROPN
dude            | Tag: NOUN
Shawn           | Tag: PROPN
friends         | Tag: NOUN
day             | Tag: NOUN
